In [30]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import BaseMessage, HumanMessage
from dotenv import load_dotenv, find_dotenv
from typing import TypedDict, Annotated
from langchain_openai import ChatOpenAI
import os
from langgraph.graph.message import add_messages 
from langgraph.checkpoint.memory import MemorySaver

In [31]:
load_dotenv(find_dotenv())

True

In [32]:
model = ChatOpenAI(
    model="gpt-4o-mini",
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_TOKEN")
)

In [33]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [34]:
def chat_node(state: ChatState):
    messages = state['messages']
    response = model.invoke(messages)
    return {'messages': [response]}

In [35]:
checkpointer = MemorySaver()

graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

chatbot = graph.compile(checkpointer= checkpointer)

In [36]:
thread_id = '1'

while True:
    user_message = input("Type here: ")

    print("User: ", user_message)

    if user_message.strip().lower() in ['exit', 'bye', 'quit']:
        break

    config = {
        'configurable': {
            'thread_id': thread_id
        }
    }
    response = chatbot.invoke({'messages': [HumanMessage(content=user_message)]}, config= config)

    print("AI: ", response['messages'][-1].content)

User:  hi my name is nitish
AI:  Hi Nitish! How can I assist you today?
User:  what is my name?
AI:  Your name is Nitish. How can I help you today?
User:  bye


In [37]:
chatbot.get_state(config= config)

StateSnapshot(values={'messages': [HumanMessage(content='hi my name is nitish', additional_kwargs={}, response_metadata={}, id='384a800a-65c2-49fe-8203-424acc9eec36'), AIMessage(content='Hi Nitish! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 12, 'prompt_tokens': 13, 'total_tokens': 25, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 15, 'engine_ttft_ms': 41, 'engine_ttlt_ms': 226, 'pre_inference_ms': 116, 'service_tbt_ms': 16, 'service_ttft_ms': 342, 'service_ttlt_ms': 521, 'total_duration_ms': 415, 'user_visible_ttft_ms': 227}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_27599ce29d', 'id': 'chatcmpl-E6J9ovEkiJwXlQSeXKyh9bxfWKPNB', 'serv